In [26]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from sklearn.svm import SVC

In [27]:
df = pd.read_csv("/Users/sidhaanthkapoor/MentalHealthClassifier/data/raw/Combined_Data.csv")
df.head()

,Unnamed: 0,statement,status
0,0,oh my gosh,Anxiety
1,1,"trouble sleeping, confused mind, restless hear...",Anxiety
2,2,"All wrong, back off dear, forward doubt. Stay ...",Anxiety
3,3,I've shifted my focus to something else but I'...,Anxiety
4,4,"I'm restless and restless, it's been a month n...",Anxiety


In [29]:
df = df.dropna(subset=["statement"])
df["statement"] = df["statement"].astype(str)
df = df[df["statement"].str.strip() != ""]
df = df.dropna(subset=["status"])

In [30]:
df["status"].value_counts()

status
Normal                  16343
Depression              15404
Suicidal                10652
Anxiety                  3841
Bipolar                  2777
Stress                   2587
Personality disorder     1077
Name: count, dtype: int64

In [31]:
X = df["statement"]
y = df["status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [40]:
vectorizer = TfidfVectorizer(
    max_features=50000,        
    ngram_range=(1, 2),        
    sublinear_tf=True,        
    min_df=3,                 
    max_df=0.95,          
    lowercase=True,            
    strip_accents="unicode",   
    token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z]+\b"
)
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

In [41]:
model = MLPClassifier(hidden_layer_sizes=(256,128), activation="relu",solver="adam",learning_rate_init=0.001, max_iter=50, random_state=42, verbose=True)

In [51]:
model.fit(X_train_vectorized, y_train)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:698: UserWarning: Training interrupted by user.
  warnings.warn("Training interrupted by user.")


MLPClassifier(hidden_layer_sizes=(256, 128), max_iter=50, random_state=42,
              verbose=True)

In [49]:
predictions = model.predict(X_test_vectorized)

In [50]:
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.7592


In [41]:
print(classification_report(y_test, predictions))

                      precision    recall  f1-score   support

             Anxiety       0.76      0.75      0.75       768
             Bipolar       0.79      0.71      0.75       556
          Depression       0.67      0.72      0.69      3081
              Normal       0.87      0.92      0.89      3269
Personality disorder       0.66      0.59      0.62       215
              Stress       0.60      0.50      0.55       517
            Suicidal       0.65      0.57      0.61      2131

            accuracy                           0.74     10537
           macro avg       0.71      0.68      0.69     10537
        weighted avg       0.74      0.74      0.74     10537

